## Analysis of cluster data produced by mPIXdaq
                                                    Günter Quast, June 2026

--- 

Sample analysis of *mPIXdaq* output files with cluster properties:

  - *.csv* format with cluster properties only
  - *.yml* format containing cluster properties and pixel indices and energies

This code serves to analyze the properties of pixel clusters derived from *miniPIX* 
frames with methods of the `class frameAnalyzer` of the Python package *mPIXdaq*.

The features of clusters and the methods to derive them are described in the 
documents *README* and *EducatorsGuide* of the *mPIXdaq* package. 

In addition to this notebook, sample input data are provided by the package *mPIXdaq* in the 
subdirectory `data/`:

- data recorded with a low-activity stone (Uranium ore) from the Black Forest
  - BlackForestStone.csv.gz  *# properties of 20'000 clusters 
  - BlackForestStone_clusters.yml.gz  *# properties and pixel indices and energies of 2'000 clusters*
  - gammaRadiation_clusters.yml.gz # *# stone shielded with 3mm of Aluminum blocking α and β radiation*  

- data from long run with ambient radiation
  - ambientRadiation_clusters.yml.gz"  *# 2 h recording time*  

- data from a sample of Radon decay products on a paper towel collected with a vacuum cleaner
  - Radon_clusters.yml.gz *# 1 h collection time, 2h recording time*  

The structure of this *Jupyter* notebook is as follows:

 1. Data import to *pandas* data frame
 2. Overview of features and data quality checks
 3. Classification of clusters
 4. Study of α particles
 5. Studies of β radiation
 6. Studies of γ interactions
 7. Identification of tracks from cosmic muons
 8. Rates and energies of α, β, γ radiation 
 9. Access to and detailed analysis of pixel data
10. Landau distribution of pixel energies along β tracks  
  

**Methods and produced cluster features of the class `frameAnalyzer`** 

From the documentation of the class class `frameAnalyzer`: 

``` 
Class frameAnalyzer:
    Analyze frame data

    - find clusters using scipy.ndimage.label()
    - compute cluster energies
    - compute position and covariance matrix of x- and y-coordinates
    - compute covariance matrix of the energy distribution
    - analyze cluster shape (using eigenvalues of covariance matrix)
    - construct a tuple with cluster properties

    Note: this algorithm only works if clusters do not overlap!

    Args:

    - 2d-frame from the miniPIX

    __call__() method Returns:

    - self.pixel_clusters: list of tuples with properties per cluster: mean of x and y
      coordinates, the number of pixels, energy, eigenvalues of covariance matrix and
      their orientation as an angle in range [-pi/2, pi/2] and the minimal and maximal
      eigenvalues of the covariance matrix of the energy distribution. The format is:
      ( (x, y), n_pix, energy, e_mx, (x_mn, y_mn, w, h), (var_mx, var_mn), angle, 
      (xEm, yEm), (varE_mx, varE_mn) )

    - self.pixel_lists: list of pixel indices contributing to each cluster, where
      pixel index is x + 256 * y
```

These properties per cluster are exported to a file in *yaml* or *csv* format 
containing the cluster features:  

**time** **x_mean** **y_mean** **n_pix** **energy** **e_mx** **x_mn** **y_mn** **w**
  **h** **var_mx** **var_mn** **angle** **xE_mean** **yE_mean** **varE_mx**  **varE_mn**

    time    : time since start of daq when frame was recorded
    x_mean  : mean x-position of cluster (in pixel numbers)
    y_mean  : mean y-position of cluster (in pixel numbers)
    n_pix   : number of pixels in cluster
    energy  : energy of cluster (= sum of pixel energies) in keV
    e_mx    : maximum pixel energy
    x_mn    : minimum x of bounding box
    y_mn    : minimum y of bounding box
    w       : width of bounding box
    h       : height of bounding box    
    var_mx  : maximum variance of geometrical cluster shape (in pixels)
    var_mn  : minimum variance of geometrical cluster shape (in pixels)
    angle   : orientation of cluster (0 = along x-axis, pi/2 = along y-axis)
    xE_mean : mean x of energy distribution  (in pixel numbers)
    yE_mean : mean y of energy distribution  (in pixel numbers)
    varE_mx : maximum variance of energy distribution  
    varE_mn : minimum variance of energy distribution 



The *yaml* file also contains a list with indices and energies of the pixels belonging 
to each of the clusters:

  > [ [idx, e],  ..., [idx, e] ], ..., [ [idx, e], ..., [idx, e] ],

  where idx = x + 256 * y  for pixel (x, y) and e is the pixel energy. 

---

In [ ]:
# necessary imports
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import pandas as pd
import yaml
import gzip

plt.style.use("default")
plt.style.use('ggplot')

### 1. Read data
---

In [ ]:
# set input file name

# data recorded with a low-activity stone (Uranium ore) from the Black Forest
# filename = "../data/BlackForestStone.csv.gz"  # properties of 20'000 clusters
filename = "../data/BlackForestStone_clusters.yml.gz"

# 3mm aluminum shielding of BlackForestStone
# filename = "../data/gammaRadiation_clusters.yml.gz"

# long run with ambient radiation and sensor oriented under 45 deg. w.r.t. the vertical direction
# filename = "../data/ambientRadiation_clusters.yml.gz"  *# 2h recording time*

# Radon collected on a paper towel with a vacuum cleaner (1h collection time, 2h recording)
# filename = "../data/Radon_clusters.yml.gz"


# *==* import data in a pandas data frame
def read_data(fn):
    global input_dict
    # function to read data from filename fn
    if '.csv' in fn:  # from csv file
        df = pd.read_csv(fn)
        meta_data = None
    elif '.yml' in fn:  # or from file in yaml format
        f = gzip.open(fn, 'rb') if fn.split('.')[-1] == 'gz' else open(fn, 'r')
        print(" .... importing yaml, be patient .... ", end='')
        input_dict = yaml.load(f, Loader=yaml.CLoader)
        keys = input_dict['keys']
        meta_data = input_dict["meta_data"]
        #  get cluter data: 1st is list of cluster properties, 2nd is list of [pixel index, energy] pairs
        cluster_data = input_dict['cluster_data']
        cluster_properties = [cluster_data[i][0] for i in range(len(cluster_data))]
        pixel_lists = [cluster_data[i][1] for i in range(len(cluster_data))]
        df = pd.DataFrame(data=cluster_properties, columns=keys)
        df['pixels'] = pixel_lists
    else:
        print("!!! unknown file format")
        df = None
    print("done")
    return meta_data, df


meta_data, df = read_data(filename)

pixels_present = True if 'pixels' in df.keys() else False

# *==* print meta data
if meta_data is not None:
    print("\n*==* Data from file " + filename)
    s_time = f"{input_dict['meta_data']['time']}"
    s_frames = f"{input_dict['eor_data']['Nframes']} frames of {input_dict['meta_data']['acq_time']}s exposure time"
    s_device = f"{input_dict['deviceInfo']['dn']}"
    print("  written on " + s_time + " with device " + s_device)
    print("  " + s_frames)
    print("  cluster features:\n  ", input_dict['keys'])

#### 1.1 Add/fix missing cluster features 

The code cell below fixes some problems in older versions of the data format.

In [ ]:
# find maximum and minimum pixel energy (only if not yet in data frame)
if pixels_present:
    if 'e_mx' not in df.keys():
        print("analyzing cluster data  -> add bounding box")
        _emx = np.zeros(len(df))
        for _i in range(len(df)):
            _emx[_i] = np.max(np.array(df['pixels'].iloc[_i])[:, 1])  # list of [px_index, energy]
        df['e_mx'] = _emx

    # add coordinates of bounding box enclosing cluster
    if 'x_mn' not in df.keys():
        print("                        -> add maximum pixel energy")
        # cluster bounding boxes as (x_mn, y_mn, w, h)
        x_mn = np.zeros(len(df))
        y_mn = np.zeros(len(df))
        w = np.zeros(len(df))
        h = np.zeros(len(df))
        for _i in range(len(df)):  # pixel lists [px_index, energy]
            x_mn[_i] = (np.array(df['pixels'].iloc[_i])[:, 0] % 256).min()
            y_mn[_i] = (np.array(df['pixels'].iloc[_i])[:, 0] // 256).min()
            w[_i] = (np.array(df['pixels'].iloc[_i])[:, 0] % 256).max() - x_mn[_i]
            h[_i] = (np.array(df['pixels'].iloc[_i])[:, 0] // 256).max() - y_mn[_i]
        df['x_mn'] = x_mn
        df['y_mn'] = y_mn
        df['w'] = w
        df['h'] = h

# fix width/height error in vers < v1.1.1 (min is 0, not 1):
if df['w'].min() == 0:
    print("fixing min(w), min(h) == 0")
    df['w'] = df['w'] + 1
    df['h'] = df['h'] + 1

#### 1.2 Add useful variables to data frame

In [ ]:
# *==* add some (useful) derived quantities
# mean energy per pixel
df['Epix_mean'] = df['energy'] / df['n_pix']
#  - circularity defined as the ratio of the smaller and the larger eigenvalue
#    of the covariance matrix of the pixels in a cluster
df['circularity'] = df['var_mn'] / np.maximum(df['var_mx'].to_numpy(), 0.001)
#  - flatness (of energy distribution) as the ratio of maximum variances
#    of pixel and energy distributions in clusters
df['flatness'] = df['varE_mx'] / np.maximum(df['var_mx'].to_numpy(), 0.001)
#  - straightness
df['straightness'] = df[['w', 'h']].max(axis=1) / df['n_pix']

### 2. Properties of data-set and overview of features in pandas data frame
---

#### 2.1 Determine number of frames in data and clean-up data

Remove empty records and - optionally - noise in single clusters at the detection threshold.

In [ ]:
# determine number of frames in data set
n_frames = len(set(df['time']))  # number of different time stamps to count frames
if meta_data is not None:
    acq_time = meta_data['acq_time']  # exposure time per frame
    T_alive = n_frames * acq_time
else:
    T_alive = df['time'].to_numpy()[-1]

# remove empty records (i.e. frames without any cluster)
is_not_nan = df['x_mean'].notna()
n_empty = (~is_not_nan).sum()
if n_empty > 0:
    print(f"removing empty records: {n_empty} removed")
    df = df[is_not_nan]

# determine total number of clusters in data set
n_clusters = len(df)
if meta_data is not None:
    print(f"\n*==* read {n_clusters} clusters in {n_frames} frames of {acq_time} s exposure time")
print(f"    -> rate = {n_clusters / T_alive:.3g} objecs/s")

# further cleaning (necessary if data are suffering from low-energy noise)
remove_noise = False
if remove_noise:
    print("      removing  pixel data ....")
    is_noise = (df['n_pix'] == 1) & (df['energy'] <= 5)
    print(f"!**! removing {is_noise.sum()} noise hits")
    df = df[~is_noise]

#### 2.2 Display data frame

In [ ]:
# *==* show data
display(df)

#### 2.3 Histograms of cluster features

In [ ]:
def plt_keys(df):
    for key in df.keys():
        if key == 'pixels':
            continue
        _ = plt.hist(df[key][df[key] != np.nan], bins=80, color='darkblue', edgecolor='grey')
        plt.yscale("log")
        plt.xlabel(key)
        plt.show()


print("*==* Overview of features")
plt_keys(df)

#### 2.4 Spacial distribution of clusters

A 2d-plot of the mean cluster positions shows how uniformly the sensor was exposed to radiation.

In [ ]:
print("*==* spacial distribution")
_ = plt.hist2d(df['x_mean'], df['y_mean'], bins=(256, 256), norm=LogNorm())
plt.xlabel("pixel number x")
plt.ylabel("pixel number y")
plt.colorbar()

#### 2.5 Check arrival times

If clusters arrival times stem from a Poisson process with constant probability over time,
the time between subsequent clusters is expected to follow an exponential distribution.

Note, however, that the frame-based read-out leads to a number of objects recorded at the
same time. Furthermore, the time-resolution is limited by the readout rate of the frames. 

In [ ]:
# difference between two events
if 'time' in df.keys():
    _dT = df['time'][1:].to_numpy() - df['time'][:-1].to_numpy()
    _ = plt.hist(_dT, bins=25)
    plt.xlabel("time between successive events (s)")
    plt.yscale('log')

### 3. Classification of clusters
---

In the cells below, the selection of different cluster types is
performed based on the cluster geometry (size and shape) and the 
distribution of the energy deposits in the clusters.


#### 3.1 Selection of different types of clusters

Definition of selection masks for different cluster types:

- alphas are circular clusters with a peaking energy distribution and large average pixel energy
- betas are tracks of several pixels with low average energy loss
- single pixels or very short tracks with low average energy are mostly gammas
- long straight tracks with low ionization are muons

Technically, such classification tasks work by defining boolean arrays wich can be used
for selecting rows from a *numpy* array or *pandas* dataframe ("boolean indexing"):

>        df[msk] 
selects only those rows where the boolean array contains a `True` value.

This method is used in the following cell to define properties of clusters. 
All code cells below depend on the classification defined here.

In [ ]:
# collect parameters for selection cuts here
small_cut = 2  # small clusters
dEdx_cut = 100  #  separate alpha signatures fom others
circularity_cut = 0.4  #  round topology
flatness_cut = 0.4  # flat energy distribution
straightness_cut = 0.75  # straight track
min_ionizing_cut = 25  #  minimum-ionizing track
long_cut = 12  # long track
emx_cut = 1200  # 2500  # avoid non-linear response if pixel energy is too high

# determine the boolean masks
is_small_cluster = df['n_pix'] <= small_cut  #
is_high_dEdx = df['Epix_mean'] > dEdx_cut  # high energy loss per pixel
is_min_ionizing = df['Epix_mean'] < min_ionizing_cut
is_circular = df['circularity'] >= circularity_cut  # circular shape
is_flat = df['flatness'] > flatness_cut  # flat energy distribution
is_straight = df['straightness'] > straightness_cut  # a straight track
is_long_track = (df['w'] >= long_cut) | (df['h'] >= long_cut)  # a long track
is_saturating = df['e_mx'] > emx_cut

# loose definition of alpha candidate by shape
shape_is_alpha = is_circular & ~is_flat
# a loose definition of an alpha
is_cand_alpha = shape_is_alpha | is_high_dEdx
# a tight definition of an alpha
is_alpha = shape_is_alpha & is_high_dEdx
# tight alpha with precise energy
is_clean_alpha = is_alpha & ~is_saturating

# definition of beta candidates
shape_is_beta = ~shape_is_alpha & (df['n_pix'] >= 5)
is_beta = shape_is_beta & ~is_high_dEdx

# define gamma candidates (low-multiplicity clusters with small dEdx)
is_cand_gamma = (df['n_pix'] < 5) & ~is_high_dEdx

# define a muon signature (a long, straight and minimumm-ionizing tracks)
is_muon = is_long_track & is_straight & is_min_ionizing & (df['e_mx'] < 50)

# define diagonal vs. tracks in x/x direction for studies of pixel energies
r_wh = 2.41  # corresponds to 22.5°    (3.73 for 15°)
is_diag = (r_wh * df['w'] > df['h']) & (r_wh * df['h'] > df['w'])

#### 3.2 2d plots for different cluster types

Looking at two variable at at time, like circularity, flatness, energy or the number of pixels, 
reveals patterns in the data and permits a visual separation of different types of signatures.
intis example, the shape information is used to flag clusters as candidates for α or β particles.  

In [ ]:
_msk = shape_is_alpha
_ = plt.hist2d(df['energy'][_msk], df['n_pix'][_msk], bins=(100, 45), norm=LogNorm())
# _ = plt.hist2d(df['energy'], df['n_pix'], bins = (100, 55), norm=LogNorm())
plt.xlim(0, 10000)
plt.ylim(0, 45)
plt.suptitle("circular & small variance")
plt.colorbar()
plt.xlabel("energy (keV)")
plt.ylabel("# pixels")
plt.text(0.8, 0.9, f"∑: {_msk.sum()}", transform=plt.gca().transAxes, color='b')
plt.show()

# define beta-candidates
_msk = shape_is_beta
_ = plt.hist2d(df['energy'][_msk], df['n_pix'][_msk], bins=(100, 75), norm=LogNorm())
plt.xlim(0, 2000)
plt.ylim(0, 75)
plt.suptitle("not (circular & small variance)")
plt.colorbar()
plt.xlabel("energy (keV)")
plt.ylabel("# pixels")
plt.text(0.8, 0.9, f"∑: {_msk.sum()}", transform=plt.gca().transAxes, color='b')
plt.show()

_msk = is_small_cluster
_ = plt.hist2d(df['energy'][_msk], df['n_pix'][_msk], bins=(100, 10), norm=LogNorm())
plt.xlim(0, 1000)
plt.ylim(0, 4)
plt.suptitle("single")
plt.colorbar()
plt.xlabel("energy (keV)")
plt.ylabel("# pixels")
plt.text(0.8, 0.9, f"∑: {_msk.sum()}", transform=plt.gca().transAxes, color='b')
plt.show()

#### Discussion

These scatter plots show a clear structure:

  - a band of small, round clusters from alpha particles that deposit all of their energy in a small volume
  - long tracks of rather low energy from tracks of beta particles
  - signatures with very few or even isolated pixels from low-energy electrons produced by gamma rays.

Note that some of the long tracks may have a significant "circularity" if they are bent !

### 4. Select and study signatures from α particles
---

In this part of the code we investigate in detail the properties of ɑ particles, 
which deposit all of their energy within a few µm of Silicon, i.e. within one pixel. 
The large amount of charge produced locally leads to leakage to neighboring pixels. 
The energy deposits of α particles with a large energy have a circular shape with 
a strong maximum in the centre. The variable "circularity", defined already above, is
very useful to quantify this property.

Because of the very localized energy deposit, the mean energy per pixel, quantified
in the variable "Epix_mean", also defined above, offers a useful distinction from long
tracks of particles with low ionization. A value greater 100 keV is a typical feature
of α particles.  

A plot of `Epix_mean` for the different types of cluster (as defined above) is produced
in the cell below.

In [ ]:
_ = plt.hist(df['Epix_mean'][shape_is_alpha], bins=100, color='r', alpha=0.5, label='ɑ')
_ = plt.hist(df['Epix_mean'][shape_is_beta], bins=100, color='b', alpha=0.5, label='β')
_ = plt.hist(df['Epix_mean'][~shape_is_beta & ~shape_is_alpha], color='y', bins=100, alpha=0.5, label='other')

plt.legend()
plt.yscale('log')
plt.xlabel("Mean pixel energy keV")
plt.show()

There is a non-linear response of the Timepix chip if the energy in a pixel
exceeds ~1200 keV. Such clusters should be rejected if a reliable measurement 
of the energy is required. Note that this removes clusters where high-energetic
α particles hits the centre of a pixel, and ɑ particles with energies above 4.8 MeV 
cannot be detected any more even if the energy is distributed over four pixels.

If a precise measurement of the energy is not needed and only detection is relevant, 
value of `emx_cut` on the maximum pixel energy may be increased. 

In [ ]:
_ = plt.hist2d(df['energy'][is_clean_alpha], df['n_pix'][is_clean_alpha], bins=(100, 45), norm=LogNorm())
plt.text(0.9, 0.9, f"∑: {is_clean_alpha.sum()}", transform=plt.gca().transAxes, color='b')
plt.xlabel("alpha energy [keV]")
plt.ylabel("number of pixels")
plt.colorbar()

A band band of α particles is obtained, with energies below 4 MeV.
The number of pixels per cluster (which is proportional to the cluster area), 
rises almost linearly with the cluster energy. 

Finally, after the background-free selection of α particles, their energy
spectrum can be plotted, as shown below. 

Note that a thin α source is needed to observe the expected fixed energy, 
because otherwise the α particles from the inner of the source lose much
of their initial energy in the source. Usually, studies of the energy loss 
of α particles in air are demonstrated with an open, thin Am-241 source. 

In [ ]:
_ = plt.hist(df['energy'][is_clean_alpha], rwidth=0.75, edgecolor='grey', color='r', alpha=0.75, bins=50)
plt.xlabel("alpha energy [keV]")
plt.title("energy spectrum of clean α")

# plot some reference lines
# plt.vlines([5310, 6020, 6090, 6770, 7680, 8770], 0.0, 1.0, colors='red')
#           Po210 Po218 Bi212 Po216 Po214 Po212

### 5. Select beta tacks and study their properties
---

We now concentrate on the worm-like topology of long traces from particles 
with small ionization. The selection masks were already defined above.
The relevant ones are:

    is_high_dEdx = df['Epix_mean'] > dEdx_cut  # high energy loss per pixel
    is_circular = df['circularity'] >= circularity_cut  # circular shape
    is_flat = df['flatness'] > flatness_cut  # flat energy distribution
    shape_is_alpha = is_circular & ~is_flat
    shape_is_beta = ~shape_is_alpha & (df['n_pix'] >= 5)

In [ ]:
_ = plt.hist(df['Epix_mean'][shape_is_beta], rwidth=0.75, bins=50, color='b')
plt.xlabel("mean energy/pixel [keV]")
plt.yscale('log')
plt.title("mean pixel energy for long non-α tracks (electrons)")

The distribution shows that the long tracks have a low mean pixel energy, 
clearly separating them from the α particles. The separation becomes perfect
if, in addition, a low value of the mean pixel energy is requested: 

    # select clean betas by also requiring low dEdx
    is_beta = shape_is_beta & ~is_high_dEdx


### 6. Plot energy spectrum of γ interactions
---

High-energetic γ rays in silicon mostly produce electron tracks via the Compton-process. 
The energies are typically - but not always - small.
An unambiguous separation of γ and β radiation is not possible. Here, we select 
clusters not classified as α or β tracks as γ candidates. 

If the *miniPIX* detector is shielded from α and β radiation the γ selection criterea
should be relaxed to include also electron signatures. 


In [ ]:
plt.style.use("default")

is_gamma = ~is_alpha & ~is_beta  # anything that is not alpha or beta candidate

_bins = np.linspace(0, 300, 50 + 1, endpoint=True)
_ = plt.hist(df['Epix_mean'][is_gamma], rwidth=0.75, bins=_bins, color='y')
plt.xlabel("mean energy/pixel [keV]")
plt.yscale('linear')
plt.title("mean pixel energy for γ")
plt.show()

_bins = np.linspace(0, 300, 50 + 1, endpoint=True)
_ = plt.hist(df[is_gamma]['energy'], bins=_bins, rwidth=0.85, color='y', label="γ")  # gamma energies
plt.xlabel("deposited energy [keV]")
plt.yscale('linear')
plt.title("Energy deposits from γ")
plt.legend()

### 7. Straight tracks as muon candidates
---

When running the data acquisition for a long time, some long, straight tracks may strike you.
These tracks have the same color as electron tracks, but do not show the typical wiggles and bends 
of electrons. To investigate further, we quantify such tracks by comparing the number of pixels 
with the diagonal of the bounding box surrounding the cluster, given by the minimum and maximum 
x- and y-coordinates. 

The property "straightness", intended to be able to select long, straight tracks, was already 
extracted above and added to the data frame.
In fact, there is a particle which is expected to produce such straight tracks in materials.
These are muons from cosmic rays that reach the ground at approximately one per minute and cm².
These particles are 200 times heavier than electrons and typically have very high energies
of tens of MeV to GeV. They therefore do not scatter, and their ionization is minimal, with
average energy deposits per pixel not exceeding 20 keV. Other than electrons, the energy loss in 
material relative to the particle energy is negligible, and therefore the ionization is
constant along the track. 

We use this information to define a muon tag:
  - long trace
  - straight
  - low mean ionization loss
  - no large energy deposits (contrary to stopping betas at the end of their range)

A very useful quantity for the identification of muons is the maximal pixel energy "e_mx" 
in a  cluster. How to extract these and add them to the data frame had already been shown
above.

The definition of the muon selection mass from above was:

    ```
    # define a muon signature (a long, straight and minimumm-ionizing tracks)
    is_muon = is_long_track & is_straight & is_min_ionizing & (df['e_mx'] < 50)
    ```

Note that there remains some ambiguity, as high-energy electrons traversing the detector 
have almost the same signature. 

In [ ]:
if 'straightness' in df.keys():
    _ = plt.hist(df[is_muon]['Epix_mean'], rwidth=0.75, bins=50, color='darkgreen')
    plt.xlabel("energy/n_pix")
    df[is_muon].hist(column='straightness', bins=50, rwidth=0.75, color='darkgreen')

#### 8. Rates amd energies of ɑ, β, γ
---

We now use our ability to distinguish different kinds of radiation to analyze 
radioactive samples, i.e. determine particle rates and energy distributions.

#### 8.1 Rates 

The **number of events** recorded in fixed time intervals is usually flat for most sources. 
A sample of Radon decay products sampled from the air in a basement, however, contains  
short-lived decay products, and a nearly exponential decrease of the rate over about
two hours is observed.  

In [ ]:
def stacked_hists(_key, _bins):
    _vals = (df[is_clean_alpha][_key], df[is_beta][_key], df[is_cand_gamma][_key])
    _labels = ('ɑ', 'β', 'γ')
    _colors = ('r', 'b', 'y')
    return plt.hist(_vals, label=_labels, bins=_bins, color=_colors, alpha=0.75, rwidth=0.75, stacked=True)


_key = 'time'
if _key in df.keys():
    f = 2 * 10 ** int(np.log10(T_alive / 3) - 1.0)
    _bins = np.linspace(0, T_alive, int(T_alive / f) + 1)

    stacked_hists(_key, _bins)

    plt.xlabel("Time (s)")
    plt.ylabel(f"Counts / {f} s")
    plt.title("Event rate vs. time")
    plt.yscale('log')
    plt.legend()

#### 8.2 Deposited Energies

The energy here means the **deposited energy** in the active detector volume. This corresponds to the
particle energy only for ɑ. β tracks with energies greater than about 100keV traverse the active 
volume if not deflected by scattering, and high-energy γ above about 100 keV most often only transfer 
a fraction of their energy to one electron in the active volume.

In dosimetry applications, it is exactly this deposited energy per Volume which is of prime interest.

The distributions of deposited energies for ɑ, β and γ radiation and their average rates
is shown below. A robust estimate of the mean energy is determined from the trimmed mean
of the energies. 

In [ ]:
from scipy.stats import trim_mean

_key = 'energy'

if _key in df.keys():
    # calculate mean energies as 10% trimmed mean
    E_alpha = trim_mean(df[is_clean_alpha][_key], 0.1)
    E_beta = trim_mean(df[is_beta][_key], 0.1)
    E_gamma = trim_mean(df[is_cand_gamma][_key], 0.1)
    energy_txt = f"energies ɑ: {E_alpha:.3g} keV, β: {E_beta:.3g} keV, γ: {E_gamma :.3g} keV"

    # histogram energy distributions
    # mx = max(df[is_clean_alpha][_key])
    mx = 4500
    f = 10 ** int(np.log10(mx / 5))
    _bins = np.linspace(0, mx, int(mx / f) + 1)
    _rc_hist = stacked_hists(_key, _bins)

    # extract number of entries from histograms
    N_alpha = (_rc_hist[0][0]).sum()
    N_beta = (_rc_hist[0][1]).sum()
    N_gamma = (_rc_hist[0][2]).sum()
    rate_txt = f"rates ɑ: {N_alpha / T_alive:.3g} Hz, β: {N_beta / T_alive:.3g} Hz, γ: {N_gamma / T_alive:.3g} Hz"

    plt.xlabel("Cluster energy (keV)")
    plt.ylabel(f"Counts / {f} keV")
    plt.text(0.3, 0.75, rate_txt, transform=plt.gca().transAxes)
    plt.text(0.3, 0.70, energy_txt, transform=plt.gca().transAxes)
    plt.title("Energy")
    plt.yscale('log')
    plt.legend()

### 9. Detailed analysis of pixel data
---

In order to improve the understanding of the recorded signatures, it is helpful to also consider the information from the individual pixels contributing to each cluster.

As an example of how to access this information we simply print the pixel lists of the first 10 alpha and beta tracks.

In [ ]:
if pixels_present:
    print("* --- pixel lists of first 10 alphas")
    print(df['pixels'][shape_is_alpha].iloc[0:10])

    print("* --- pixel lists of first 10 beta tracks")
    print(df['pixels'][is_beta].iloc[0:10])

#### 9.1 Access and plot individual pixel energies

With a bit more programming effort, access to individual pixels is possible.

In [ ]:
if pixels_present:
    pxl_Elst_a = []  # collect pixel energies for alphas
    for _px_lst in df['pixels'][is_clean_alpha]:
        pxl_Elst_a += [_l[1] for _l in _px_lst]

    pxl_Elst_b = []  # collect pixel energies for betas
    for _px_lst in df['pixels'][is_beta]:
        pxl_Elst_b += [_l[1] for _l in _px_lst]

    pxl_Elst_o = []  # collect pixel energies for others
    for _px_lst in df['pixels'][~is_beta & ~is_alpha]:
        pxl_Elst_o += [_l[1] for _l in _px_lst]

    e_mx = 2000
    _bins = np.linspace(0, e_mx, 200 + 1, endpoint=True)
    _ = plt.hist(pxl_Elst_a, rwidth=0.85, bins=_bins, color='r', label="α", alpha=0.33)
    _ = plt.hist(pxl_Elst_b, rwidth=0.85, bins=_bins, color='b', label="β", alpha=0.33)
    plt.xlabel("pixel energy [keV]")
    plt.yscale('log')
    plt.legend()
    plt.title("Pixel energies")
    plt.show()

    _ = plt.hist(pxl_Elst_o, rwidth=0.85, bins=_bins, label="not α, β", color='y', alpha=0.5)
    plt.xlabel("pixel energy [keV]")
    plt.yscale('log')
    plt.legend()
    plt.title("Pixel energies")
    plt.show()

#### 9.2 Graphical displays of pixel maps

For a detailed understanding of typical signatures of particles, in particular
in the overlap region between β traces and γ-induced electrons, or between
long electron and muon tracks, it is helpful to visually inspect the pixel clusters. 

First, we import a function to plot the energy maps of pixel clusters.

In [ ]:
from mpixdaq.mpixhelpers import plot_cluster

The code below uses the masks for tagging certain signatures, as discussed above, 
and shows some pixel maps.

By selecting the relevant masks, you can convince yourself of the properties
of α and  β particles, find muons in your data sample, or study
other signatures like the properties of electrons induced by γ rays. 

In [ ]:
# define selection mask
any = is_beta | ~is_beta

mask = any  # take a random sample of first clusters

# examples of selection masks
# mask = is_beta & (df['energy'] < 100)  # low-energy beta
# mask = is_clean_alpha & (df['energy'] < 1000)  # low-energy alpha
# mask = is_cand_beta & (df['n_pix'] > 50)   # long tracks
# mask = ~is_cand_alpha & ~is_beta  # other signatures, mostly gamma-induced or alpha/beta/gamma overlap
# mask = is_muon  # & (df['n_pix'] > 20)

# masks for special events
##mask = is_clean_alpha & (df['energy'] < 500) # low-energy alpha with alpha-shape
##mask = (df['energy'] < 500) & df[is_highdEdx] # low energy, high-ionizing
##mask = is_clean_alpha & (df['n_pix'] >8) # special alpha
##mask = df['Epix_mean']> 80 & (df['energy'] < 750) # low-energy alpha

# get pixel coordinates and energies
if pixels_present:
    ioff = 0
    Nmx = 10
    mx_count = min(Nmx + ioff, len(df['pixels'][mask]))  # number of images

    # set plotting style with dark
    plt.style.use("dark_background")

    print(f"*--- {mask.sum()} clusters fulfill selection")
    print(f"      showing images of first {mx_count} pixel clusters\n")
    for _idx in range(ioff, ioff + mx_count):
        _px_l = df['pixels'][mask].iloc[_idx]  # list of [px_index, energy]
        _c_prp = df[mask].iloc[_idx]  # cluster properties
        print(f" {_idx} -> {_c_prp['n_pix']:.0f} pixels, energy {_c_prp['energy']:.0f} keV")
        # print(_px_l)
        _fig = plot_cluster(_px_l, _idx)
        plt.show()

plt.style.use('default')

### 10 Landau distribution of pixel energies along β tracks 
---

The **energy deposits per pixel** fluctuate around the mean energy. 
The distribution is asymmetric with a long tail towards high values 
and can be parameterized by a Landau probability density function.

To show this, long, straight β tracks running along the x or y
directions are selected below and projected on the x or y axes.
This selection uses the masks `is_beta`, `is_diag` and `is_long_track`
defined above.
The end point (where the electrons slow down) is identified by
large values of the energy deposit per pixel. More than four pixels 
away from the stopping point, the mean energy per pixel approaches 
a constant value expected for minimum-ionizing particles. 

The resulting distribution of pixel energies with an overlaid
fit of a Landau distribution is produced by the code below. 

Since the selection of β tracks is very tight, this analysis ideally 
needs a large sample of clusters recorded with a β source like Sr-90. 

In [ ]:
from mpixdaq.mpixhelpers import pxl2map
from LandauFit import nLandau, fit_Landau
from PhyPraKit import histstat

if pixels_present:
    # collect pixel energies for betas
    pixel_energies = []  # collect pixel energies for betas
    track_energies = []

    for _px_l in df['pixels'][is_beta & is_straight & ~is_diag & is_long_track]:
        # unpack clusters into an array (use unpacking in plot_cluster)
        _cmap = pxl2map(_px_l)
        h, w = np.shape(_cmap)

        # project on 1-d to get linear tracks and store linearized tracks in list
        if w > h:  # project on x
            _px_e = [_cmap[:, i].sum() for i in range(w)]
        elif h > w:  # project on y
            _px_e = [_cmap[i, :].sum() for i in range(h)]
        # check if track has an high-ionizing end
        if (_px_e[0] + _px_e[1]) / 2.0 > 35:
            pixel_energies.append(_px_e)
            track_energies.append(sum(_px_e))
        elif (_px_e[-1] + _px_e[-2]) / 2.0 > 35:  # invert list order
            _ipx_e = [_px_e[i - 1] for i in range(len(_px_e), 0, -1)]
            pixel_energies.append(_ipx_e)
            track_energies.append(sum(_ipx_e))

    n_tracks = len(track_energies)
    print(f"found {n_tracks} long tracks along x or y")

    if n_tracks > 5:
        print("Performing analysis of pixel energies")
        # plot energy in pixels:
        e_mx = 40.0
        e_mn = 5.0
        _bins = np.linspace(e_mn, e_mx, int(e_mx - e_mn) + 1)
        _xplt = np.linspace(e_mn, e_mx, 10 * int(e_mx - e_mn))

        # list of pixel energies more than 4 pixels away from stopping point
        _pxe = []
        for j in range(4, long_cut):
            _pxe += [pixel_energies[i][j] for i in range(n_tracks)]
        plt.title("Distribution of pixel energies of long, straight β tracks")
        _bc, _be, _ = plt.hist(_pxe, _bins, rwidth=0.85, label=f"pixels 5-{long_cut}", alpha=0.5)
        mean, sigma, sigma_mean = histstat(_bc, _be, pr=True)
        pvals, perrs, gof, pnams = fit_Landau(_bc, _be, mean, sigma)
        _ = plt.plot(_xplt, nLandau(_xplt, *pvals), label='Landau')
        plt.xlabel("energy per pixel (keV)", size='large')
        plt.ylabel("entries / keV", size='large')
        plt.legend()
        plt.show()
    else:
        print("not enough long, straight tracks found for pixel analysis")